# Topic Modeling
## This notebook outlines the concepts involved in Topic Modeling


Topic modeling is a statistical model to **discover** the abstract "topics" that occur in a collection of documents

It is commonly used in text document. But nowadays, in social media analysis, topic modeling is an emerging research area.

One of the most popular algorithms used is **Latent Dirichlet Allocation** which was proposed by
David Blei et al in 2003.

Dataset:
https://raw.githubusercontent.com/subashgandyer/datasets/main/kaggledatasets.csv

### Steps
- Install the necessary library
- Import the necessary libraries
- Download the dataset
- Load the dataset
- Pre-process the dataset
    - Tokenize
    - Stop words removal
    - Non-alphabetic words removal
    - Lowercase
- Create a dictionary for the document
- Filter low frequency words
- Create an Index to word dictionary
- Train the Topic Model
- Predict on the dataset
- Visualize the topics

### Install the necessary library

In [ ]:
%%capture
! pip install gensim

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Import the necessary libraries

In [ ]:
from pprint import pprint

import pandas as pd

import gensim
from gensim import corpora, models
from gensim.models import LdaModel
from gensim.corpora import Dictionary

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from nltk.stem.wordnet import WordNetLemmatizer

nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Download the dataset

In [ ]:
!wget https://raw.githubusercontent.com/subashgandyer/datasets/main/kaggledatasets.csv

--2025-11-04 02:09:33--  https://raw.githubusercontent.com/subashgandyer/datasets/main/kaggledatasets.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3881130 (3.7M) [text/plain]
Saving to: ‘kaggledatasets.csv.1’

kaggledatasets.csv. 100%[===================>]   3.70M  --.-KB/s    in 0.09s   

2025-11-04 02:09:33 (39.8 MB/s) - ‘kaggledatasets.csv.1’ saved [3881130/3881130]



### Load the dataset

In [ ]:
df = pd.read_csv("kaggledatasets.csv")
df.head()

,Title,Subtitle,Owner,Votes,Versions,Tags,Data Type,Size,License,Views,Download,Kernels,Topics,URL,Description
0,Credit Card Fraud Detection,Anonymized credit card transactions labeled as...,Machine Learning Group - ULB,1241,"Version 2,2016-11-05|Version 1,2016-11-03",crime\nfinance,CSV,144 MB,ODbL,"442,136 views","53,128 downloads","1,782 kernels",26 topics,https://www.kaggle.com/mlg-ulb/creditcardfraud,The datasets contains transactions made by cre...
1,European Soccer Database,"25k+ matches, players & teams attributes for E...",Hugo Mathien,1046,"Version 10,2016-10-24|Version 9,2016-10-24|Ver...",association football\neurope,SQLite,299 MB,ODbL,"396,214 views","46,367 downloads","1,459 kernels",75 topics,https://www.kaggle.com/hugomathien/soccer,The ultimate Soccer database for data analysis...
2,TMDB 5000 Movie Dataset,"Metadata on ~5,000 movies from TMDb",The Movie Database (TMDb),1024,"Version 2,2017-09-28",film,CSV,44 MB,Other,"446,255 views","62,002 downloads","1,394 kernels",46 topics,https://www.kaggle.com/tmdb/tmdb-movie-metadata,Background\nWhat can we say about the success ...
3,Global Terrorism Database,"More than 170,000 terrorist attacks worldwide,...",START Consortium,789,"Version 2,2017-07-19|Version 1,2016-12-08",crime\nterrorism\ninternational relations,CSV,144 MB,Other,"187,877 views","26,309 downloads",608 kernels,11 topics,https://www.kaggle.com/START-UMD/gtd,"Context\nInformation on more than 170,000 Terr..."
4,Bitcoin Historical Data,Bitcoin data at 1-min intervals from select ex...,Zielak,618,"Version 11,2018-01-11|Version 10,2017-11-17|Ve...",history\nfinance,CSV,119 MB,CC4,"146,734 views","16,868 downloads",68 kernels,13 topics,https://www.kaggle.com/mczielinski/bitcoin-his...,Context\nBitcoin is the longest running and mo...


### Explore the dataset

In [ ]:
df.columns

Index(['Title', 'Subtitle', 'Owner', 'Votes', 'Versions', 'Tags', 'Data Type',
       'Size', 'License', 'Views', 'Download', 'Kernels', 'Topics', 'URL',
       'Description'],
      dtype='object')

In [ ]:
df.dtypes

,0
Title,object
Subtitle,object
Owner,object
Votes,int64
Versions,object
Tags,object
Data Type,object
Size,object
License,object
Views,object


In [ ]:
df['Description'].iloc[0]

"The datasets contains transactions made by credit cards in September 2013 by european cardholders. This dataset presents transactions that occurred in two days, where we have 492 frauds out of 284,807 transactions. The dataset is highly unbalanced, the positive class (frauds) account for 0.172% of all transactions.\nIt contains only numerical input variables which are the result of a PCA transformation. Unfortunately, due to confidentiality issues, we cannot provide the original features and more background information about the data. Features V1, V2, ... V28 are the principal components obtained with PCA, the only features which have not been transformed with PCA are 'Time' and 'Amount'. Feature 'Time' contains the seconds elapsed between each transaction and the first transaction in the dataset. The feature 'Amount' is the transaction Amount, this feature can be used for example-dependant cost-senstive learning. Feature 'Class' is the response variable and it takes value 1 in case o

In [ ]:
df.isnull().sum()

,0
Title,0
Subtitle,104
Owner,0
Votes,0
Versions,5
Tags,542
Data Type,0
Size,0
License,0
Views,5


### Extract the data for topic modeling

In [ ]:
for i in df['Description'].items():
    raw = str(i[1]).lower()
    print(raw)

Output hidden; open in https://colab.research.google.com to view.

### Pre-process the dataset
- Tokenize
- Stop words removal
- Non-alphabetic words removal
- Lowercase
- Define them

### Define the pattern, tokenizer, stop words and lemmatizer

In [ ]:
pattern = r'\b[^\d\W]+\b'
tokenizer = RegexpTokenizer(pattern)
en_stop = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

### Preprocess

In [ ]:
texts = []

for idx, desc in df['Description'].items():
    # clean and tokenize document string
    raw_text = str(desc).lower()
    tokens = tokenizer.tokenize(raw_text)

    # remove stop words
    stopped_tokens = [t for t in tokens if t not in en_stop]

    # lemmatize tokens
    lemma_tokens = [lemmatizer.lemmatize(t) for t in stopped_tokens]

    # remove single-character words
    filtered_tokens = [t for t in lemma_tokens if len(t) > 1]

    # add tokens to list
    texts.append(filtered_tokens)

# check one example
print(texts[0])

['datasets', 'contains', 'transaction', 'made', 'credit', 'card', 'september', 'european', 'cardholder', 'dataset', 'present', 'transaction', 'occurred', 'two', 'day', 'fraud', 'transaction', 'dataset', 'highly', 'unbalanced', 'positive', 'class', 'fraud', 'account', 'transaction', 'contains', 'numerical', 'input', 'variable', 'result', 'pca', 'transformation', 'unfortunately', 'due', 'confidentiality', 'issue', 'cannot', 'provide', 'original', 'feature', 'background', 'information', 'data', 'feature', 'principal', 'component', 'obtained', 'pca', 'feature', 'transformed', 'pca', 'time', 'amount', 'feature', 'time', 'contains', 'second', 'elapsed', 'transaction', 'first', 'transaction', 'dataset', 'feature', 'amount', 'transaction', 'amount', 'feature', 'used', 'example', 'dependant', 'cost', 'senstive', 'learning', 'feature', 'class', 'response', 'variable', 'take', 'value', 'case', 'fraud', 'otherwise', 'given', 'class', 'imbalance', 'ratio', 'recommend', 'measuring', 'accuracy', 'usi

### Create a dictionary

In [61]:
dictionary = Dictionary(texts)
dictionary[1]

'account'

### Filter low frequency words

In [ ]:
dictionary.filter_extremes(no_below=10, no_above=0.5)
# convert tokenized documents into a document-term matrix
corpus = [dictionary.doc2bow(text) for text in texts]
print(corpus)

[[(0, 3), (1, 1), (2, 2), (3, 3), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 3), (13, 2), (14, 1), (15, 1), (16, 1), (17, 1), (18, 3), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 7), (31, 1), (32, 4), (33, 1), (34, 1), (35, 1), (36, 3), (37, 1), (38, 1), (39, 1), (40, 1), (41, 1), (42, 2), (43, 1), (44, 1), (45, 1), (46, 1), (47, 1), (48, 2), (49, 1), (50, 1), (51, 1), (52, 1), (53, 1), (54, 1), (55, 1), (56, 1), (57, 1), (58, 1), (59, 1), (60, 1), (61, 1), (62, 1), (63, 1), (64, 1), (65, 1), (66, 1), (67, 1), (68, 1), (69, 1), (70, 1), (71, 1), (72, 1), (73, 1), (74, 2), (75, 1), (76, 7), (77, 1), (78, 1), (79, 1), (80, 1), (81, 1), (82, 1), (83, 1), (84, 2)], [(9, 1), (10, 1), (12, 2), (23, 1), (28, 1), (30, 1), (31, 1), (36, 3), (42, 1), (43, 1), (52, 2), (55, 3), (57, 1), (60, 2), (61, 1), (74, 2), (82, 2), (83, 1), (85, 2), (86, 2), (87, 1), (88, 1), (89, 1), (90, 1), (91, 3), (92, 1), (93, 1),

### Create an index to word dictionary

In [62]:
temp = dictionary[0]  # This is only to "load" the dictionary.
id2word = dictionary.id2token

### Train the Topic model

### What is LDA (Latent Dirichlet Allocation)?

**LDA** is a topic modeling algorithm that automatically discovers **topics** hidden in a collection of documents.


- Each **document** is made up of several **topics**.  
- Each **topic** is made up of certain **words** that often appear together.

Example:  
A document might be **70% "machine learning"** and **30% "healthcare"**,  
and the "machine learning" topic might include words like *model, data, algorithm*.


### How it works (in plain terms)

1. **Start with random guesses** of which topics belong to which words.  
2. For each word, update the topic assignment based on:
   - how common the topic is in the document  
   - how common the word is in that topic  
3. Repeat many times.  
4. Gradually, the model finds stable patterns where topics and words fit well.


**In short:**  
> LDA learns which words group together into topics  
> and how much of each topic is present in every document.


In [63]:
ldamodel = LdaModel(corpus, num_topics=15, id2word = id2word, passes=20)

### Display the topics

In [ ]:
pprint(ldamodel.top_topics(corpus,topn=5))

[([(np.float32(0.015720382), 'file'),
   (np.float32(0.011163963), 'http'),
   (np.float32(0.010989649), 'csv'),
   (np.float32(0.007671345), 'contains'),
   (np.float32(0.007087897), 'com')],
  np.float64(-0.9941452179024873)),
 ([(np.float32(0.03345198), 'text'),
   (np.float32(0.024020923), 'word'),
   (np.float32(0.020386241), 'language'),
   (np.float32(0.019431308), 'speech'),
   (np.float32(0.017333686), 'corpus')],
  np.float64(-1.3066421243029427)),
 ([(np.float32(0.031081242), 'model'),
   (np.float32(0.024423623), 'image'),
   (np.float32(0.019049956), 'trained'),
   (np.float32(0.0132447025), 'feature'),
   (np.float32(0.012674279), 'pre')],
  np.float64(-1.359354026943187)),
 ([(np.float32(0.026515888), 'restaurant'),
   (np.float32(0.024789527), 'year'),
   (np.float32(0.0193515), 'review'),
   (np.float32(0.016200973), 'city'),
   (np.float32(0.015434203), 'information')],
  np.float64(-1.428446937503389)),
 ([(np.float32(0.011719833), 'year'),
   (np.float32(0.008455161

### Display the 15 topics with words

In [ ]:
for idx in range(15):
    print("Topic #%s:" % idx, ldamodel.print_topic(idx, 10))

Topic #0: 0.019*"column" + 0.013*"crime" + 0.012*"number" + 0.011*"variable" + 0.011*"day" + 0.011*"activity" + 0.011*"per" + 0.010*"value" + 0.009*"weather" + 0.009*"year"
Topic #1: 0.032*"de" + 0.027*"team" + 0.023*"question" + 0.018*"player" + 0.017*"vote" + 0.016*"news" + 0.013*"percentage" + 0.013*"la" + 0.013*"point" + 0.012*"per"
Topic #2: 0.429*"university" + 0.080*"state" + 0.054*"college" + 0.024*"california" + 0.022*"texas" + 0.017*"institute" + 0.013*"north" + 0.012*"new" + 0.011*"florida" + 0.011*"technology"
Topic #3: 0.047*"csv" + 0.041*"row" + 0.024*"facility" + 0.018*"student" + 0.016*"education" + 0.016*"gas" + 0.015*"california" + 0.015*"url" + 0.014*"lat" + 0.014*"column"
Topic #4: 0.031*"model" + 0.024*"image" + 0.019*"trained" + 0.013*"feature" + 0.013*"pre" + 0.011*"learning" + 0.010*"recognition" + 0.010*"research" + 0.009*"contains" + 0.009*"network"
Topic #5: 0.022*"weapon" + 0.018*"back" + 0.015*"customer" + 0.015*"solar" + 0.015*"center" + 0.014*"service" + 

## LSI Model

**LSI**, also called **LSA (Latent Semantic Analysis)**, is a method that finds **hidden relationships between words and documents** by using **math (linear algebra)** instead of probabilities.

- Each **document** can be represented as a vector of words.  
- Some words mean similar things and often appear together.  
- LSI reduces the data to a smaller set of **concepts (topics)** that capture those hidden patterns.


### How it works

1. Build a **document–term matrix** (counts of each word in each document).  
2. Apply **Singular Value Decomposition (SVD)**, a mathematical technique that breaks the matrix into three smaller matrices:
   - documents × concepts  
   - concepts × words  
3. Keep only the top *k* concepts (topics).  
4. These concepts represent the main themes in the data.


**In short:**  
> LSI uses math (SVD) to uncover hidden themes,  
> showing which documents and words are related through shared meaning —  
> not just exact word matches.


In [64]:
from gensim.models import LsiModel
lsamodel = LsiModel(corpus, num_topics=10, id2word = id2word)
pprint(lsamodel.print_topics(num_topics=10, num_words=10))

[(0,
  '0.970*"developer" + 0.174*"convicted" + 0.076*"week" + 0.051*"simply" + '
  '0.049*"txt" + 0.039*"four" + 0.031*"formation" + 0.028*"damage" + '
  '0.027*"unesco" + 0.027*"become"'),
 (1,
  '0.389*"holy" + 0.248*"match" + 0.221*"station" + 0.200*"accessible" + '
  '0.177*"obtained" + 0.173*"ultimate" + 0.159*"financial" + 0.156*"except" + '
  '0.147*"constantly" + 0.126*"affiliated"'),
 (2,
  '0.436*"holy" + 0.307*"station" + 0.259*"match" + -0.250*"extracting" + '
  '-0.225*"serve" + 0.175*"affiliated" + 0.173*"constantly" + '
  '-0.154*"ultimate" + 0.151*"michael" + -0.133*"except"'),
 (3,
  '0.595*"extracting" + 0.535*"serve" + 0.263*"different" + 0.261*"drop" + '
  '0.119*"complicated" + 0.116*"holy" + -0.098*"ultimate" + -0.093*"financial" '
  '+ 0.090*"station" + -0.088*"except"'),
 (4,
  '-0.402*"financial" + 0.325*"without" + 0.266*"max" + 0.198*"url" + '
  '0.192*"worst" + 0.186*"interest" + 0.180*"cleaning" + 0.174*"found" + '
  '0.170*"goal" + 0.165*"quantity"'),
 (5

In [65]:
for idx in range(10):
    print("Topic #%s:" % idx, lsamodel.print_topic(idx, 10))
print("=" * 20)

Topic #0: 0.970*"developer" + 0.174*"convicted" + 0.076*"week" + 0.051*"simply" + 0.049*"txt" + 0.039*"four" + 0.031*"formation" + 0.028*"damage" + 0.027*"unesco" + 0.027*"become"
Topic #1: 0.389*"holy" + 0.248*"match" + 0.221*"station" + 0.200*"accessible" + 0.177*"obtained" + 0.173*"ultimate" + 0.159*"financial" + 0.156*"except" + 0.147*"constantly" + 0.126*"affiliated"
Topic #2: 0.436*"holy" + 0.307*"station" + 0.259*"match" + -0.250*"extracting" + -0.225*"serve" + 0.175*"affiliated" + 0.173*"constantly" + -0.154*"ultimate" + 0.151*"michael" + -0.133*"except"
Topic #3: 0.595*"extracting" + 0.535*"serve" + 0.263*"different" + 0.261*"drop" + 0.119*"complicated" + 0.116*"holy" + -0.098*"ultimate" + -0.093*"financial" + 0.090*"station" + -0.088*"except"
Topic #4: -0.402*"financial" + 0.325*"without" + 0.266*"max" + 0.198*"url" + 0.192*"worst" + 0.186*"interest" + 0.180*"cleaning" + 0.174*"found" + 0.170*"goal" + 0.165*"quantity"
Topic #5: 0.535*"except" + -0.436*"financial" + -0.193*"ac

## Visualize the topics and documents with the trained Topic Model
- Use pyLDAvis from gensim

In [66]:
%%capture
!pip install pyLDAvis

In [67]:
import pyLDAvis.gensim

### Enable the notebook for visualization

In [68]:
import warnings
warnings.filterwarnings("ignore")

In [69]:
pyLDAvis.enable_notebook()

### Visualize the Topic model

### What is λ (Lambda) in pyLDAvis?

The **λ slider** controls how the most relevant words for each topic are chosen.

- **λ = 1.0** → shows the most **frequent** words in that topic.  
- **λ = 0.0** → shows the most **unique or exclusive** words to that topic.  
- **λ ≈ 0.6 (default)** → balances between frequency and uniqueness.

Move λ **left** to see what makes the topic special,  
move it **right** to see the most common words within the topic.


In [70]:
pyLDAvis.gensim.prepare(ldamodel, corpus, dictionary)

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
8      0.235090  0.000004       1        1  39.380036
12     0.210936 -0.102763       2        1  31.058057
14     0.118116 -0.000947       3        1   4.046986
1     -0.021783  0.081242       4        1   3.865594
2      0.125391 -0.033181       5        1   3.836327
13     0.047435 -0.062198       6        1   3.396939
0     -0.046721  0.109987       7        1   2.639369
10    -0.005162 -0.022965       8        1   2.473465
7      0.042351  0.010370       9        1   2.089896
11     0.025522  0.042848      10        1   1.525927
9     -0.026405  0.064180      11        1   1.425215
4     -0.140042  0.065532      12        1   1.142425
3     -0.232578 -0.299606      13        1   1.120850
6     -0.158223  0.063634      14        1   1.102839
5     -0.173928  0.083863      15        1   0.896073, topic_info=                Term         Freq        Total Category  logprob  loglift
579        developer   894.000000   894.000000  Default  30.0000  30.0000
566        convicted   826.000000   826.000000  Default  29.0000  29.0000
200             holy   532.000000   532.000000  Default  28.0000  28.0000
603           except  1018.000000  1018.000000  Default  27.0000  27.0000
772        worldwide   596.000000   596.000000  Default  26.0000  26.0000
...              ...          ...          ...      ...      ...      ...
1428  characteristic    13.118442   205.078469  Topic15  -5.0359   1.9655
761            usage    12.877113   240.853262  Topic15  -5.0544   1.7862
347          costing    11.431038   395.112602  Topic15  -5.1735   1.1721
180              far    10.837400   451.281734  Topic15  -5.2269   0.9858
516       accessible     7.904633  1224.290778  Topic15  -5.5424  -0.3278

[899 rows x 6 columns], token_table=      Topic      Freq     Term
term                          
1549      1  0.213979  absence
1549      3  0.454705  absence
1549      5  0.227353  absence
1549      9  0.066868  absence
1549     11  0.026747  absence
...     ...       ...      ...
515       9  0.162191     zero
515      11  0.007912     zero
515      14  0.027691     zero
2057      1  0.192777   zillow
2057      4  0.819303   zillow

[2161 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[9, 13, 15, 2, 3, 14, 1, 11, 8, 12, 10, 5, 4, 7, 6])